# Lesson 38 — Offline Salary Verification Replay

## Contents

- [Research question and method](#research-method)
- [Saved snapshot and provider salary provenance](#snapshot-provenance)
- [Observed quotations and current parser](#current-parser)
- [Historical comparison and synthetic controls](#historical-controls)
- [Observed quotations before and after Lesson 37](#observed-comparison)
- [Infrastructure evaluation](#infrastructure-evaluation)

<a id="research-method"></a>

## Research question and method

Did Lesson 37 improve deterministic parser recognition for salary quotations in the saved Lesson 36 Adzuna sample?

Load the saved response, identify descriptions containing a pound sign, and manually select one salary quotation from each matching description. Give both parsers identical strings. Use separate synthetic controls to isolate changes in format recognition and numerical validation.

Run from the `notebooks/` directory with pandas and the project's Python package installed. Keep the Lesson 36 snapshot in `data/raw/` and ensure Git can resolve `f2f2f86` and `5f37e1f`. The comparison imports the parser from the working checkout, which at Lesson 38 is based on `5f37e1f`. Replay uses only local inputs.

<a id="snapshot-provenance"></a>

## Saved snapshot and provider salary provenance

In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
pd.set_option("display.max_colwidth", None)

In [ ]:
# Locate the saved Adzuna response.
data_path = Path("../data/raw/adzuna_lesson36_20261116T092204Z.json")

# Load the JSON into a Python dictionary.
with data_path.open(encoding="utf-8") as file:
    data = json.load(file)

# Convert the advertisement records into a DataFrame.
jobs = pd.DataFrame(data["results"])
jobs["salary_is_predicted"] = pd.to_numeric(
    jobs["salary_is_predicted"]
)

print(f"Advertisements loaded: {len(jobs)}")
print(f"Columns available: {len(jobs.columns)}")

jobs.head()

In [ ]:
salary_columns = [
    "title",
    "salary_min",
    "salary_max",
    "salary_is_predicted",
]

jobs[salary_columns].head(10)

In [ ]:
jobs["salary_is_predicted"].value_counts(dropna=False)

In [ ]:
jobs[salary_columns].isna().sum()

In [ ]:
jobs.loc[
    :,
    ["title", "description"]
].head(5)

In [ ]:
for index in [0, 1]:
    print(f"Advertisement {index}: {jobs.loc[index, 'title']}")
    print(jobs.loc[index, "description"][:250])
    print("-" * 60)

In [ ]:
jobs["contains_gbp"] = jobs["description"].str.contains(
    "£",
    regex=False,
    na=False,
)

jobs["contains_gbp"].value_counts()

In [ ]:
pd.crosstab(
    jobs["salary_is_predicted"],
    jobs["contains_gbp"],
    rownames=["Predicted salary"],
    colnames=["Pound sign in description"],
    margins=True,
)

In [ ]:
non_predicted_without_gbp = (
    (jobs["salary_is_predicted"] == 0)
    & (~jobs["contains_gbp"])
)

predicted_with_gbp = (
    (jobs["salary_is_predicted"] == 1)
    & jobs["contains_gbp"]
)

exceptions = jobs.loc[
    non_predicted_without_gbp | predicted_with_gbp,
    [
        "title",
        "salary_min",
        "salary_max",
        "salary_is_predicted",
        "description",
    ],
]

exceptions

### Provider flags and evidence in saved descriptions

The cross-tabulation contains four rows where the provider flag and the pound-sign indicator differ from the simple expectation that a non-predicted salary has a textual quote. Two predicted rows contain salary ranges, and two non-predicted rows have no pound sign in the saved descriptions. The flags describe provider provenance, while the symbol check describes what is visible in the excerpts.

In [ ]:
for index in [9, 15]:
    print(f"Advertisement {index}: {jobs.loc[index, 'title']}")
    print(f"Provider salary: £{jobs.loc[index, 'salary_min']:,.2f}")
    print(f"Description: {jobs.loc[index, 'description'][:200]}")
    print("-" * 60)

In [ ]:
comparison = pd.DataFrame({
    "advertisement": [9, 15],
    "advertised_min": [75000, 55000],
    "advertised_max": [85000, 65000],
})

comparison["provider_prediction"] = jobs.loc[
    comparison["advertisement"], "salary_min"
].to_numpy()

comparison["above_advertised_max"] = (
    comparison["provider_prediction"] - comparison["advertised_max"]
)

comparison

In [ ]:
for index, job in jobs.loc[jobs["contains_gbp"]].iterrows():
    print(f"\nAdvertisement {index}: {job['title']}")
    print(job["description"][:300])
    print("-" * 60)

In [ ]:
description = jobs.loc[6, "description"]

position = description.find("£")

print("Pound sign position:", position)
print(description[max(0, position - 60):position + 120])

### Salary evidence and provider estimates

Our initial exploration examined 20 saved Adzuna job advertisements collected on 16 November 2026.

**Provider salary provenance**

- 12 advertisements (60%) contained provider-predicted salary values.
- 8 advertisements (40%) contained non-predicted provider salary values.
- All 20 records contained provider salary minimum and maximum values.
- Only 8 descriptions (40%) contained an explicit sterling symbol (`£`).

The presence of provider salary values does not establish that those salaries are supported by evidence in the advertisement.

**Discrepancies between advertised and predicted salaries**

Two advertisements contained explicit salary ranges that differed substantially from their provider predictions:

| Advertisement | Advertised range | Provider prediction | Difference above upper bound |
|---|---|---|---|
| Lead Data Engineer (row 9) | £75,000–£85,000 | £96,114.48 | £11,114.48 |
| Data Engineer (row 15) | £55,000–£65,000 | £81,696.02 | £16,696.02 |

Both provider values were flagged as predicted. The cause of these discrepancies is unknown.

Two further advertisements contained non-predicted provider salary values without sterling references in their saved descriptions.

**Variation in salary evidence formats**

Manual inspection of the eight descriptions containing `£` identified several formats:

- Salary ranges with full pound amounts.
- A range expressed using `to` rather than a hyphen.
- A decimal daily-rate range.
- A single daily rate.
- An approximate salary expressed using `Circa`.
- A salary range followed by `pa`.

None of the inspected quotations used the abbreviated `k` notation introduced in Lesson 37.

**Limitations and next steps**

These findings describe a small, saved sample and cannot be generalised to the wider UK labour market.

The presence of a salary quotation does not guarantee that it satisfies the deterministic verification contract. In particular, several quotations lack an explicit salary period.

The replay below measures recognition of selected strings. Evidence selection and complete verification remain outside this comparison.

<a id="current-parser"></a>

## Observed quotations and current parser

Read the eight manually selected quotations from the saved descriptions and compare the parser outputs with two synthetic positive controls.

In [ ]:
from uk_job_market_agents.agents.salary_review import parse_salary_range

evidence_samples = {
    0: "£450.00 - £500.00 per day",
    1: "£500 per day",
    6: "£95,000 - £105,000",
    7: "£70,000 to £90,000",
    9: "£75,000 - £85,000",
    15: "£55,000 - £65,000",
    16: "Circa £55,000",
    17: "£76,000 - £95,000 pa",
}

assert set(evidence_samples) == set(jobs.index[jobs["contains_gbp"]])

for index, evidence in evidence_samples.items():
    assert evidence in jobs.loc[index, "description"], (index, evidence)
    result = parse_salary_range(evidence)
    print(f"Advertisement {index}: {result}")

In [ ]:
control_samples = {
    "Full amounts": "£45,000 - £55,000 per year",
    "Abbreviated amounts": "£45k - £55k per year",
}

for label, evidence in control_samples.items():
    result = parse_salary_range(evidence)
    print(f"{label}: {result}")

### Initial parser results

The current parser recognised 0 of 8 observed quotations. The two synthetic positive controls both returned `(45000, 55000)`. The historical comparison reuses the same inputs to distinguish existing support from changes introduced in Lesson 37.

<a id="historical-controls"></a>

## Historical comparison and synthetic controls

Use Git to inspect the Lesson 37 change and recover the pre-change pattern from `f2f2f86`. The notebook reads the regex literal and flags from the historical source and reproduces the displayed function’s whole-string match and integer conversion. This reconstruction covers the historical range parser only. Both parsers receive the same synthetic strings.

In [ ]:
import subprocess

result = subprocess.run(
    [
        "git",
        "log",
        "-8",
        "--oneline",
        "--",
        "src/uk_job_market_agents/agents/salary_review.py",
    ],
    capture_output=True,
    text=True,
    check=True,
    cwd="..",
)

print(result.stdout)

In [ ]:
result = subprocess.run(
    [
        "git",
        "diff",
        "f2f2f86",
        "5f37e1f",
        "--",
        "src/uk_job_market_agents/agents/salary_review.py",
    ],
    capture_output=True,
    text=True,
    check=True,
    cwd="..",
)

print(result.stdout)

In [ ]:
result = subprocess.run(
    [
        "git",
        "show",
        "f2f2f86:src/uk_job_market_agents/agents/salary_review.py",
    ],
    capture_output=True,
    text=True,
    check=True,
    cwd="..",
)

historical_source = result.stdout

start = historical_source.index("def parse_salary_range(")
end = historical_source.find("\ndef ", start + 1)

if end == -1:
    end = len(historical_source)

print(historical_source[start:end])

In [ ]:
import ast
import re

historical_tree = ast.parse(historical_source)

pattern_assignment = next(
    node
    for node in historical_tree.body
    if isinstance(node, ast.Assign)
    and any(
        isinstance(target, ast.Name)
        and target.id == "SALARY_RANGE_PATTERN"
        for target in node.targets
    )
)

historical_pattern = ast.literal_eval(
    pattern_assignment.value.args[0]
)

historical_regex = re.compile(
    historical_pattern,
    getattr(re, pattern_assignment.value.args[1].attr),
)

def historical_parse_salary_range(evidence: str) -> tuple[int, int] | None:
    """Mirror the bounds conversion in the pre-Lesson 37 function."""
    match = historical_regex.fullmatch(evidence.strip())

    if match is None:
        return None

    salary_min = int(match.group(1).replace(",", ""))
    salary_max = int(match.group(2).replace(",", ""))

    return salary_min, salary_max

In [ ]:
for label, evidence in control_samples.items():
    before = historical_parse_salary_range(evidence)
    after = parse_salary_range(evidence)

    print(f"{label}")
    print(f"  Before: {before}")
    print(f"  After:  {after}")

In [ ]:
boundary_samples = {
    "Decimal k": "£45.5k - £55.5k per year",
    "Mixed notation": "£45,000 - £55k per year",
    "Uppercase K": "£45K - £55K per year",
    "Reversed range": "£55k - £45k per year",
    "Zero lower bound": "£0k - £55k per year",
    "Fractional pound": "£45.0001k - £55k per year",
}

for label, evidence in boundary_samples.items():
    before = historical_parse_salary_range(evidence)
    after = parse_salary_range(evidence)

    print(f"{label}")
    print(f"  Before: {before}")
    print(f"  After:  {after}")

In [ ]:
validation_samples = {
    "Reversed full-pound range": "£55,000 - £45,000 per year",
    "Zero lower bound": "£0 - £55,000 per year",
    "Equal positive bounds": "£45,000 - £45,000 per year",
}

for label, evidence in validation_samples.items():
    before = historical_parse_salary_range(evidence)
    after = parse_salary_range(evidence)

    print(f"{label}")
    print(f"  Before: {before}")
    print(f"  After:  {after}")

### Synthetic control results

**Format recognition**

- Full-pound salary ranges were accepted by both implementations.
- Abbreviated `k` notation, decimal `k` notation, uppercase `K` and mixed full-pound/abbreviated ranges were accepted only by the current implementation.
- Invalid abbreviated ranges, including reversed bounds, zero lower bounds and fractional-pound amounts, were rejected by the current parser.

**Numerical validation**

The historical parser accepted reversed and zero-valued ranges expressed in full-pound notation.

The current parser rejected both cases while preserving support for equal positive bounds.

These controls isolate parser behaviour for the formats tested. They are excluded from the eight-quotation coverage calculation.

<a id="observed-comparison"></a>

## Observed quotations before and after Lesson 37

In [ ]:
comparison_rows = []

for index, evidence in evidence_samples.items():
    before = historical_parse_salary_range(evidence)
    after = parse_salary_range(evidence)

    comparison_rows.append({
        "advertisement": index,
        "evidence": evidence,
        "before_lesson_37": before,
        "after_lesson_37": after,
        "changed": before != after,
    })

real_world_comparison = pd.DataFrame(comparison_rows)

for column in ("before_lesson_37", "after_lesson_37"):
    recognised = real_world_comparison[column].notna().sum()
    print(f"{column}: {recognised}/{len(real_world_comparison)} recognised")
print(f"Changed results: {real_world_comparison['changed'].sum()}")

# Check the longer wording for row 15 separately from the primary comparison.
row_15_with_currency = evidence_samples[15] + " GBP"
assert row_15_with_currency in jobs.loc[15, "description"]
print(
    "Row 15 with trailing GBP:",
    historical_parse_salary_range(row_15_with_currency),
    parse_salary_range(row_15_with_currency),
)

real_world_comparison

The eight manually selected salary quotations from the saved Adzuna snapshot were evaluated using both the historical and current deterministic salary-range parsers.

**Results**

- Historical parser: 0 of 8 quotations recognised (0%).
- Current parser: 0 of 8 quotations recognised (0%).
- Observed change in recognition coverage: 0 percentage points.

No differences in parser output were observed for these eight quotations.

**Interpretation**

Lesson 37 demonstrably improved support for abbreviated salary notation and strengthened numerical validation in controlled synthetic tests. However, those improvements did not increase recognition coverage in this small real-world sample.

The observed quotations contain alternative separators, period abbreviations, single salaries, approximate amounts, decimal daily rates and ranges without explicit salary periods.

These formats represent potential areas for future parser development, subject to explicit verification and data-quality requirements.

**Limitations**

The comparison uses manually selected evidence strings from 8 of 20 saved advertisements. It does not measure automatic evidence extraction, complete salary verification outcomes or the prevalence of these formats across the wider UK labour market.

Row 15 uses the range without its trailing `GBP` label in the primary comparison. A separate check including the label also returned `None` for both parsers.

<a id="infrastructure-evaluation"></a>

## Infrastructure evaluation

This informal evaluation exercised local snapshot loading, inspection of provider salary metadata and deterministic parser replay using Git history. It exposed gaps in recognition and showed that provider metadata can differ from the pay quoted in saved descriptions.

A full end-to-end integration test would also exercise retrieval or snapshot ingestion and orchestration, automatic evidence extraction, verification of the extracted values and period against the advert, and entry into the analytical dataset.